# 03 - Statistical Analysis and Sex-Interaction ANCOVA

This notebook runs the primary lipid association models and ANCOVA interaction analysis.

## Core models

- Per-lipid OLS by cohort: `lipid ~ SI_avg + niareagansc + age_death`
- Category-mean OLS by cohort
- ANCOVA interaction on full cohort: `lipid ~ SI_avg * msex + niareagansc + age_death`

## Script equivalent

The same workflow is available via `scripts/03_statistical_analysis.py`.


In [ ]:
import pandas as pd

from config import DATA_PROCESSED_DIR, FINAL_FORMATTED_FILENAME, TABLES_DIR, ensure_project_dirs
from src.data_utils import get_lipid_columns
from src.stats_utils import (
    RegressionSpec,
    compute_category_means,
    run_ancova_sex_interaction,
    run_per_lipid_regression,
    split_by_sex,
)

ensure_project_dirs()


## Step 1: Load processed dataset and define model specification


In [ ]:
df = pd.read_csv(DATA_PROCESSED_DIR / FINAL_FORMATTED_FILENAME)
lipid_cols = get_lipid_columns(df)

spec = RegressionSpec(
    predictors=("SI_avg", "niareagansc", "age_death"),
    primary_predictor="SI_avg",
    min_n=20,
)

print("Dataset shape:", df.shape)
print("Lipid columns:", len(lipid_cols))


## Step 2: Run sex-stratified and full-cohort per-lipid OLS models


In [ ]:
cohorts = split_by_sex(df)
for cohort_name, cohort_df in cohorts.items():
    lipid_results = run_per_lipid_regression(cohort_df, lipid_columns=lipid_cols, spec=spec)
    lipid_results.to_csv(TABLES_DIR / f"stats_lipid_{cohort_name}.csv", index=False)

    category_df = compute_category_means(cohort_df, lipid_columns=lipid_cols)
    category_cols = [c for c in category_df.columns if c.startswith("catmean_")]
    category_results = run_per_lipid_regression(category_df, lipid_columns=category_cols, spec=spec)
    category_results.to_csv(TABLES_DIR / f"stats_category_{cohort_name}.csv", index=False)

print("Saved stats_lipid_* and stats_category_* tables.")


## Step 3: Run ANCOVA sex-interaction models on full cohort


In [ ]:
ancova_lipid = run_ancova_sex_interaction(
    df=df,
    lipid_columns=lipid_cols,
    main_predictor="SI_avg",
    sex_column="msex",
    covariates=("niareagansc", "age_death"),
    min_n=20,
)
ancova_lipid.to_csv(TABLES_DIR / "ancova_sex_lipid.csv", index=False)

category_all = compute_category_means(df, lipid_columns=lipid_cols)
category_cols_all = [c for c in category_all.columns if c.startswith("catmean_")]
ancova_category = run_ancova_sex_interaction(
    df=category_all,
    lipid_columns=category_cols_all,
    main_predictor="SI_avg",
    sex_column="msex",
    covariates=("niareagansc", "age_death"),
    min_n=20,
)
ancova_category.to_csv(TABLES_DIR / "ancova_sex_category.csv", index=False)

print("ANCOVA lipid rows:", len(ancova_lipid))
print("ANCOVA category rows:", len(ancova_category))
ancova_lipid.head(12)


## Interpreting ANCOVA output

- `coef_interaction` is the SI-by-sex interaction effect size.
- `p_interaction` is the nominal p-value for sex difference in SI association.
- `fdr_p_interaction` is the multiple-testing-adjusted interaction p-value.

Low adjusted interaction p-values indicate lipids where the SI-lipid slope differs between males and females.


## Next notebook

Run `notebooks/04_sensitivity_no_ad.ipynb` to repeat the analysis after excluding AD pathology cases.